### Лабораторная работа №7. Генерация текста посредством нейросетевых методов на основе данных о рассматриваемой предметной области

### Предобработка данных
Импорт необходимых библиотек для работы с глубоким обучением, массивами и файловой системой. Определение базовых гиперпараметров модели и загрузка исходного набора данных по предметной области «Учет телефонных переговоров»

In [8]:
import tensorflow as tf
import numpy as np
import os
import pickle
import tqdm
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
from string import punctuation

# Определение базовых гиперпараметров согласно методике
sequence_length = 100
BATCH_SIZE = 128
EPOCHS = 50
FILE_PATH = "telecom_dataset.txt"
BASENAME = os.path.basename(FILE_PATH)

# Чтение исходного текста
text = open(FILE_PATH, encoding="utf-8").read()

# Очистка и нормализация текста (нижний регистр + удаление пунктуации)
text = text.lower()
text = text.translate(str.maketrans("", "", punctuation))

print(f"Пример очищенного текста (первые 200 символов):\n{text[:200]}...")

Пример очищенного текста (первые 200 символов):
учет телефонных переговоров является важнейшей задачей современной телефонной компании телефонная компания предоставляет своим абонентам выделенные телефонные линии которые используются для осуществле...


### Формирование словарей кодирования символов
Нейронная сеть работает исключительно с числами, поэтому мы извлекаем все уникальные символы (буквы, пробелы), встречающиеся в тексте, и создаем два словаря: `char2int` (для перевода символа в число) и `int2char` (для обратного перевода числа в символ при генерации текста). Эти словари сохраняются на диск

In [9]:
# Получение списка уникальных символов
n_chars = len(text)
vocab = ''.join(sorted(set(text)))

n_unique_chars = len(vocab)
print("Уникальные символы (vocab):", vocab)
print("Общее количество символов в тексте:", n_chars)
print("Количество уникальных символов:", n_unique_chars)

# Создание словарей кодирования
char2int = {c: i for i, c in enumerate(vocab)}
int2char = {i: c for i, c in enumerate(vocab)}

# Сохранение словарей с помощью pickle для дальнейшего использования
pickle.dump(char2int, open(f"{BASENAME}-char2int.pickle", "wb"))
pickle.dump(int2char, open(f"{BASENAME}-int2char.pickle", "wb"))

# Кодирование всего текста (преобразование каждого символа в число)
encoded_text = np.array([char2int[c] for c in text])
print(f"\nЗакодированный текст (первые 20 символов): {encoded_text[:20]}")

Уникальные символы (vocab): 
 абвгдежзийклмнопрстуфхцчшщыьюя
Общее количество символов в тексте: 4126
Количество уникальных символов: 32

Закодированный текст (первые 20 символов): [21 25  7 20  1 20  7 13  7 22 16 15 15 28 23  1 17  7 18  7]


### Создание конвейера данных (tf.data.Dataset) и формирование последовательностей
Для эффективного обучения применяются инструменты `tf.data`. Закодированный массив разбивается на последовательности. Входные данные модели — это набор из 100 символов, а целевое значение — 1 последующий символ, который модель должна научиться предсказывать. Также применяется One-Hot кодирование.

In [10]:
# Создание датасета из закодированного массива
char_dataset = tf.data.Dataset.from_tensor_slices(encoded_text)

# Формирование "окон" для последовательностей
sequences = char_dataset.batch(2 * sequence_length + 1, drop_remainder=True)

# Функция разбиения последовательности на входы и цели
def split_sample(sample):
    ds = tf.data.Dataset.from_tensors((sample[:sequence_length], sample[sequence_length]))
    for i in range(1, (len(sample)-1) // 2):
        input_ = sample[i: i+sequence_length]
        target = sample[i+sequence_length]
        other_ds = tf.data.Dataset.from_tensors((input_, target))
        ds = ds.concatenate(other_ds)
    return ds

# Подготовка входов (inputs) и целей (targets)
dataset = sequences.flat_map(split_sample)

# Функция для One-Hot кодирования
def one_hot_samples(input_, target):
    return tf.one_hot(input_, n_unique_chars), tf.one_hot(target, n_unique_chars)

# Применение One-Hot кодирования к каждому образцу
dataset = dataset.map(one_hot_samples)

# Вывод контрольной информации о форме (shape) тензоров
for element in dataset.take(1):
    print("Входной размер (Input shape):", element[0].shape)
    print("Размер цели (Target shape):", element[1].shape)

# Подготовка финального конвейера данных: зацикливание, перемешивание и разбиение на батчи
ds = dataset.repeat().shuffle(1024).batch(BATCH_SIZE, drop_remainder=True)
print(f"Итоговый датасет готов к подаче в модель. Размер батча: {BATCH_SIZE}")

Входной размер (Input shape): (100, 32)
Размер цели (Target shape): (32,)
Итоговый датасет готов к подаче в модель. Размер батча: 128


### Архитектура нейронной сети
Построение модели рекуррентной нейронной сети. В основе лежат два слоя LSTM (Long Short-Term Memory) по 256 нейронов в каждом. Для предотвращения переобучения (overfitting) используется слой `Dropout`, который случайным образом "отключает" 30% нейронов на этапе обучения. Выходной слой `Dense` использует функцию активации `softmax` для вычисления вероятности появления каждого следующего символа из нашего словаря

In [11]:
# Создание директории для сохранения результатов (весов модели), если её нет
if not os.path.isdir("results"):
    os.mkdir("results")

# Определение пути для сохранения весов модели
model_weights_path = f"results/{BASENAME}-{sequence_length}.h5"

# Построение последовательной модели (Sequential)
model = Sequential([
    # Первый слой LSTM возвращает последовательности (return_sequences=True), чтобы следующий LSTM слой мог принять их на вход
    LSTM(256, input_shape=(sequence_length, n_unique_chars), return_sequences=True),

    # Слой регуляризации Dropout (отключает 30% нейронов)
    Dropout(0.3),

    # Второй слой LSTM
    LSTM(256),

    # Полносвязный выходной слой с количеством нейронов, равным размеру словаря
    # Softmax преобразует логиты в вероятности (сумма всех выходов равна 1)
    Dense(n_unique_chars, activation="softmax"),
])

# Вывод сводной информации об архитектуре модели
model.summary()

# Компиляция модели:
# loss="categorical_crossentropy" - функция потерь для многоклассовой классификации
# optimizer="adam" - современный и эффективный алгоритм оптимизации
# metrics=["accuracy"] - метрика для отслеживания точности
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 100, 256)       │       295,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 100, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 256)            │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         8,224 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 829,472 (3.16 MB)

 Trainable params: 829,472 (3.16 MB)

 Non-trainable params: 0 (0.00 B)

### Обучение модели и проведение экспериментов
На данном этапе происходит запуск процесса обучения нейронной сети. Модель будет обучаться на протяжении заданного количества эпох (`EPOCHS = 50`). На каждой эпохе алгоритм обратного распространения ошибки будет корректировать веса нейронов (около 829 тыс. параметров) так, чтобы минимизировать функцию потерь (loss) и повысить точность (accuracy) предсказания следующего символа. После завершения обучения веса модели будут сохранены на диск для последующей генерации текста

In [12]:
# Проверка наличия директории для сохранения результатов
if not os.path.isdir("results"):
    os.mkdir("results")

print(f"Запущен процесс обучения на количестве эпох: {EPOCHS}")

# Запуск процесса обучения
# Параметр steps_per_epoch указывает, сколько батчей нужно обработать,
# чтобы считать одну эпоху завершенной
history = model.fit(
    ds,
    steps_per_epoch=(len(encoded_text) - sequence_length) // BATCH_SIZE,
    epochs=EPOCHS
)

# Сохранение обученной модели (весов) в файл .h5
model.save(model_weights_path)
print(f"\nОбучение успешно завершено! Веса модели сохранены в: {model_weights_path}")

Запущен процесс обучения на количестве эпох: 50
Epoch 1/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 4s 35ms/step - accuracy: 0.0970 - loss: 3.1761
Epoch 2/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.1270 - loss: 3.0780
Epoch 3/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.1258 - loss: 3.0621
Epoch 4/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.1313 - loss: 3.0340
Epoch 5/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.1595 - loss: 2.9582
Epoch 6/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.1830 - loss: 2.8941
Epoch 7/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.2082 - loss: 2.7729
Epoch 8/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.2286 - loss: 2.6604
Epoch 9/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.2606 - loss: 2.5093
Epoch 10/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.3170 - loss: 2.3401
Epoch 11/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.3725 - loss: 2.0992
Epoch 12/50
31/


Обучение успешно завершено! Веса модели сохранены в: results/telecom_dataset.txt-100.h5


### Результаты генерации текста
На этом этапе мы используем обученную модель для генерации нового текста. Мы подаем нейросети небольшую начальную строку (`seed`). Модель анализирует её, предсказывает наиболее вероятный следующий символ, добавляет его к строке, сдвигает «окно» чтения на один символ вправо и повторяет процесс. Таким образом символ за символом генерируется связный текст на основе выученного контекста

In [15]:
# Загрузка словарей из файлов (согласно методике)
char2int = pickle.load(open(f"{BASENAME}-char2int.pickle", "rb"))
int2char = pickle.load(open(f"{BASENAME}-int2char.pickle", "rb"))
vocab_size = len(char2int)

# Загрузка сохраненных весов модели
model.load_weights(model_weights_path)

# Начальная строка для генерации (в нижнем регистре, без знаков препинания. Строка должна состоять из символов, которые есть в нашем словаре
seed = "каждый звонок абонента автоматически фиксируется в базе "
s = seed
n_chars = 400 # Количество символов, которые мы хотим сгенерировать
generated = ""

print("Запущен процесс генерация текста...\n")

# Цикл генерации
for i in tqdm.tqdm(range(n_chars), "Generating text"):
    # Создаем пустой тензор для входных данных
    X = np.zeros((1, sequence_length, vocab_size))

    # Выполняем One-Hot кодирование для текущей строки seed
    for t, char in enumerate(seed):
        # Если seed короче sequence_length, заполняем с конца
        X[0, (sequence_length - len(seed)) + t, char2int[char]] = 1

    # Предсказание вероятностей для следующего символа
    predicted = model.predict(X, verbose=0)[0]

    # Выбираем символ с максимальной вероятностью
    next_index = np.argmax(predicted)
    next_char = int2char[next_index]

    # Добавляем символ к результату
    generated += next_char

    # Сдвигаем seed: удаляем первый символ и добавляем предсказанный в конец
    seed = seed[1:] + next_char

print("\n\n=== РЕЗУЛЬТАТ ГЕНЕРАЦИИ ===")
print("Начальная строка (Seed):", s)
print("Сгенерированный текст (Generated text):")
print(generated)

Запущен процесс генерация текста...



Generating text: 100%|██████████| 400/400 [00:31<00:00, 12.52it/s]



=== РЕЗУЛЬТАТ ГЕНЕРАЦИИ ===
Начальная строка (Seed): каждый звонок абонента автоматически фиксируется в базе 
Сгенерированный текст (Generated text):
вонанта компании долтни меть зовоший премен начаны а никк да тожед ивутью зазачей нирка ди вотень сата зоволая диметально уречавысе ся дич ононы менита воже донети порегова ным чита волений сиккок и раделиниет сто а бонени вумату обелний сетом билеть остимь уреговора ве применисто та инывы уменьшается в зависимости от общей длительности разговора чем дольше астем заровений скидки ительно устим пол
